# Pon.Bike Demo - Semantic View & Cortex Agent

This notebook creates:
1. **Semantic View** for Cortex Analyst (natural language to SQL)
2. **Cortex Search Service** for review search
3. **Cortex Agent** for Snowflake Intelligence

In [ ]:
import os
from snowflake.snowpark import Session

session = Session.builder.config("connection_name", os.getenv("SNOWFLAKE_CONNECTION_NAME", "oregon_tp")).create()
print(f"Connected as: {session.get_current_user()}")
print(f"Role: {session.get_current_role()}")
print(f"Warehouse: {session.get_current_warehouse()}")

In [ ]:
session.sql("USE DATABASE PON_BIKE_DEMO").collect()
session.sql("USE SCHEMA ANALYTICS").collect()
print("Using PON_BIKE_DEMO.ANALYTICS")

## Step 1: Create Semantic View for Cortex Analyst

In [ ]:
semantic_view_ddl = """
CREATE OR REPLACE SEMANTIC VIEW PON_BIKE_DEMO.ANALYTICS.PON_BIKE_COMPETITIVE_INTEL

TABLES (
    BRANDS AS PON_BIKE_DEMO.ANALYTICS.DIM_BRANDS 
        PRIMARY KEY (BRAND_ID)
        COMMENT = 'Bicycle brands - our 9 Pon.Bike brands + 16 competitors',
    
    REVIEWS AS PON_BIKE_DEMO.ANALYTICS.REVIEWS 
        PRIMARY KEY (REVIEW_ID)
        COMMENT = 'Product reviews with ratings for bicycles',
    
    BRAND_METRICS AS PON_BIKE_DEMO.ANALYTICS.FACT_BRAND_METRICS 
        PRIMARY KEY (BRAND_ID, YEAR_MONTH)
        COMMENT = 'Monthly brand performance metrics in the Dutch market'
)

RELATIONSHIPS (
    REVIEWS(BRAND_ID) REFERENCES BRANDS(BRAND_ID),
    BRAND_METRICS(BRAND_ID) REFERENCES BRANDS(BRAND_ID)
)

FACTS (
    BRANDS.FOUNDED_YEAR AS FOUNDED_YEAR COMMENT = 'Year brand was founded',
    BRANDS.NL_DEALER_COUNT AS NL_DEALER_COUNT WITH SYNONYMS = ('dealers', 'stores', 'shops') COMMENT = 'Number of dealers in Netherlands',
    REVIEWS.RATING AS RATING WITH SYNONYMS = ('stars', 'score') COMMENT = 'Star rating 1-5',
    BRAND_METRICS.NL_UNITS_SOLD AS NL_UNITS_SOLD WITH SYNONYMS = ('units sold', 'bikes sold', 'sales volume') COMMENT = 'Monthly units sold in NL',
    BRAND_METRICS.NL_REVENUE_EUR AS NL_REVENUE_EUR WITH SYNONYMS = ('revenue', 'sales') COMMENT = 'Monthly revenue in EUR',
    BRAND_METRICS.AVG_SELLING_PRICE_EUR AS AVG_SELLING_PRICE_EUR WITH SYNONYMS = ('avg price', 'average price', 'ASP') COMMENT = 'Average selling price in EUR',
    BRAND_METRICS.NL_MARKET_SHARE_PCT AS NL_MARKET_SHARE_PCT WITH SYNONYMS = ('market share', 'share') COMMENT = 'Monthly NL market share percentage',
    BRAND_METRICS.ONLINE_SENTIMENT_SCORE AS ONLINE_SENTIMENT_SCORE WITH SYNONYMS = ('sentiment') COMMENT = 'Online sentiment score',
    BRAND_METRICS.NEW_MODEL_LAUNCHES AS NEW_MODEL_LAUNCHES WITH SYNONYMS = ('launches', 'new models') COMMENT = 'New bike models launched'
)

DIMENSIONS (
    BRANDS.BRAND_ID AS BRAND_ID COMMENT = 'Unique brand identifier',
    BRANDS.BRAND_NAME AS BRAND_NAME WITH SYNONYMS = ('brand', 'name') COMMENT = 'Name of the bicycle brand',
    BRANDS.PARENT_COMPANY AS PARENT_COMPANY WITH SYNONYMS = ('owner', 'group') COMMENT = 'Parent company',
    BRANDS.HEADQUARTERS AS HEADQUARTERS WITH SYNONYMS = ('country', 'HQ') COMMENT = 'Brand headquarters country',
    BRANDS.BRAND_TYPE AS BRAND_TYPE WITH SYNONYMS = ('type', 'segment') COMMENT = 'Brand type/segment',
    BRANDS.PRICE_TIER AS PRICE_TIER WITH SYNONYMS = ('pricing', 'tier') COMMENT = 'Price category',
    BRANDS.IS_PON_BRAND AS IS_PON_BRAND WITH SYNONYMS = ('our brand', 'pon brand', 'owned', 'ours', 'our brands', 'pon') COMMENT = 'TRUE = Pon.Bike brand, FALSE = competitor',
    BRANDS.PRIMARY_CATEGORY AS PRIMARY_CATEGORY COMMENT = 'Primary bike category for the brand',
    
    REVIEWS.REVIEW_ID AS REVIEW_ID COMMENT = 'Review identifier',
    REVIEWS.BIKE_CATEGORY AS BIKE_CATEGORY WITH SYNONYMS = ('category', 'bike type') COMMENT = 'Bike category reviewed',
    REVIEWS.REVIEWER_NAME AS REVIEWER_NAME COMMENT = 'Reviewer name',
    REVIEWS.REVIEWER_CITY AS REVIEWER_CITY WITH SYNONYMS = ('city', 'location') COMMENT = 'Reviewer city in Netherlands',
    REVIEWS.REVIEW_TITLE AS REVIEW_TITLE COMMENT = 'Review title',
    REVIEWS.REVIEW_TEXT AS REVIEW_TEXT WITH SYNONYMS = ('feedback', 'comment', 'review') COMMENT = 'Full review content',
    REVIEWS.PURCHASE_TYPE AS PURCHASE_TYPE COMMENT = 'New, Used, or Lease/Subscription',
    REVIEWS.USAGE_TYPE AS USAGE_TYPE WITH SYNONYMS = ('usage', 'use case') COMMENT = 'How the bike is used',
    REVIEWS.REVIEW_DATE AS REVIEW_DATE COMMENT = 'Date posted',
    
    BRAND_METRICS.YEAR_MONTH AS YEAR_MONTH WITH SYNONYMS = ('month', 'period', 'date') COMMENT = 'Month'
)

METRICS (
    REVIEWS.AVG_RATING AS AVG(REVIEWS.RATING) COMMENT = 'Average rating',
    REVIEWS.REVIEW_COUNT AS COUNT(REVIEWS.REVIEW_ID) COMMENT = 'Number of reviews',
    BRAND_METRICS.TOTAL_UNITS_SOLD AS SUM(BRAND_METRICS.NL_UNITS_SOLD) COMMENT = 'Total units sold',
    BRAND_METRICS.TOTAL_REVENUE AS SUM(BRAND_METRICS.NL_REVENUE_EUR) COMMENT = 'Total revenue in EUR',
    BRAND_METRICS.AVG_MARKET_SHARE AS AVG(BRAND_METRICS.NL_MARKET_SHARE_PCT) COMMENT = 'Average market share'
)

COMMENT = 'Competitive intelligence for Pon.Bike in the Dutch bicycle market'

AI_SQL_GENERATION 'Our brands (Pon.Bike) have IS_PON_BRAND = TRUE (9 brands: Gazelle, Cannondale, Santa Cruz, Cervelo, Kalkhoff, Focus, Urban Arrow, Veloretti, Schwinn). Competitors have IS_PON_BRAND = FALSE (16 brands). Always compare Pon brand performance against competitors when relevant. Revenue and prices are in EUR.'

AI_QUESTION_CATEGORIZATION '
categorization_rules:
  - name: "employee_data"
    description: "Questions about employee salaries, personal information, or HR data"
    response: "I cannot provide information about employee data. Please contact HR directly."
  - name: "financial_internals"
    description: "Questions about internal profit margins, cost structures, or confidential financial details"
    response: "Detailed financial internals are confidential. I can help with general revenue and performance metrics."
  - name: "competitor_secrets"
    description: "Questions asking for confidential competitor strategies or non-public information"
    response: "I can only provide analysis based on publicly available data and our own operational metrics."
  - name: "personal_reviewer_info"
    description: "Questions trying to identify or contact specific reviewers"
    response: "I cannot provide personal contact information. Reviews are shown anonymously for privacy."
'
"""

print(f"Semantic view DDL ready ({len(semantic_view_ddl)} chars)")

In [ ]:
session.sql(semantic_view_ddl).collect()
print("Semantic View created!")

## Step 2: Test Cortex Analyst

In [ ]:
import requests
import json

def ask_analyst(question: str):
    token = session.connection._rest._token
    account = session.connection.account
    host = session.connection.host
    
    url = f"https://{host}/api/v2/cortex/analyst/message"
    
    headers = {
        "Authorization": f"Snowflake Token=\"{token}\"",
        "Content-Type": "application/json"
    }
    
    payload = {
        "messages": [{"role": "user", "content": [{"type": "text", "text": question}]}],
        "semantic_view": "PON_BIKE_DEMO.ANALYTICS.PON_BIKE_COMPETITIVE_INTEL"
    }
    
    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()
    
    result = response.json()
    return result

def run_analyst_query(question: str):
    result = ask_analyst(question)
    
    sql = None
    for msg in result.get("message", {}).get("content", []):
        if msg.get("type") == "sql":
            sql = msg.get("statement")
            break
    
    if sql:
        print(f"Generated SQL:\n{sql}\n")
        return session.sql(sql).to_pandas()
    else:
        print("Response:", json.dumps(result, indent=2))
        return None

print("Cortex Analyst functions ready")

In [ ]:
result = run_analyst_query("How do our brand ratings compare to competitors?")
display(result)

## Step 3: Create Cortex Search Service for Reviews

In [ ]:
session.sql("""
CREATE OR REPLACE CORTEX SEARCH SERVICE PON_BIKE_DEMO.ANALYTICS.BIKE_REVIEWS_SEARCH
ON REVIEW_TEXT
ATTRIBUTES BRAND_ID, BIKE_CATEGORY, USAGE_TYPE
WAREHOUSE = AI_WH
TARGET_LAG = '1 day'
EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
AS (
    SELECT 
        REVIEW_ID,
        BRAND_ID,
        BIKE_CATEGORY,
        REVIEWER_NAME,
        REVIEWER_CITY,
        RATING,
        REVIEW_DATE,
        REVIEW_TITLE,
        REVIEW_TEXT,
        USAGE_TYPE
    FROM PON_BIKE_DEMO.ANALYTICS.REVIEWS
)
""").collect()
print("Cortex Search Service created!")

In [ ]:
from snowflake.core import Root

pon_brands = session.sql("""
    SELECT BRAND_ID, BRAND_NAME FROM PON_BIKE_DEMO.ANALYTICS.DIM_BRANDS 
    WHERE IS_PON_BRAND = TRUE
""").collect()
pon_brand_ids = [row['BRAND_ID'] for row in pon_brands]
print(f"Pon brands: {[row['BRAND_NAME'] for row in pon_brands]}")
print(f"BRAND_IDs: {pon_brand_ids}\n")

root = Root(session)
search_service = (root
    .databases["PON_BIKE_DEMO"]
    .schemas["ANALYTICS"]
    .cortex_search_services["BIKE_REVIEWS_SEARCH"]
)

filter_conditions = {"@or": [{"@eq": {"BRAND_ID": brand_id}} for brand_id in pon_brand_ids]}

results = search_service.search(
    query="complaints about battery range and motor noise on e-bikes",
    columns=["REVIEW_TEXT", "RATING", "REVIEWER_NAME", "REVIEW_TITLE", "BRAND_ID", "BIKE_CATEGORY"],
    filter=filter_conditions,
    limit=5
)

print("Search: 'complaints about battery range and motor noise on e-bikes' (PON BRANDS ONLY)\n")
for r in results.results:
    print(f"Rating: {r['RATING']} - {r['REVIEW_TITLE']} (Brand ID: {r['BRAND_ID']}, {r['BIKE_CATEGORY']})")
    print(f"   {r['REVIEW_TEXT'][:200]}...")
    print()

## Step 4: Create Cortex Agent

In [ ]:
session.sql("""
CREATE OR REPLACE AGENT PON_BIKE_DEMO.ANALYTICS.PON_BIKE_ANALYST
  COMMENT = 'Competitive intelligence analyst for Pon.Bike in the Dutch bicycle market'
  PROFILE = '{
    "display_name": "Pon.Bike Competitive Analyst"
  }'
  FROM SPECIFICATION $$
  {
    "models": {
      "orchestration": "claude-sonnet-4-5"
    },
    "instructions": {
      "orchestration": "You help analyze bicycle market competitive intelligence for Pon.Bike in the Netherlands. Use the analyst tool for data queries (ratings, metrics, market share, sales comparisons). Use the search tool to find specific product reviews or customer feedback. Our brands have IS_PON_BRAND=TRUE (Gazelle, Cannondale, Santa Cruz, Cervelo, Kalkhoff, Focus, Urban Arrow, Veloretti, Schwinn). Competitors have IS_PON_BRAND=FALSE.",
      "response": "Be concise and data-driven. When discussing reviews, include relevant quotes. Always specify brand names rather than just IDs."
    },
    "tools": [
      {
        "tool_spec": {
          "type": "cortex_analyst_text_to_sql",
          "name": "bike_analyst",
          "description": "Query bicycle competitive intelligence data including brand ratings, market share, revenue, units sold, and aggregated statistics"
        }
      },
      {
        "tool_spec": {
          "type": "cortex_search",
          "name": "review_search",
          "description": "Search product reviews to find specific feedback, complaints, or praise about bicycle products and brands"
        }
      }
    ],
    "tool_resources": {
      "bike_analyst": {
        "semantic_view": "PON_BIKE_DEMO.ANALYTICS.PON_BIKE_COMPETITIVE_INTEL",
        "execution_environment": {
          "type": "warehouse",
          "warehouse": "AI_WH"
        }
      },
      "review_search": {
        "search_service": "PON_BIKE_DEMO.ANALYTICS.BIKE_REVIEWS_SEARCH",
        "max_results": 10
      }
    }
  }
  $$
""").collect()
print("Cortex Agent created with Analyst + Search tools!")

In [ ]:
session.sql("GRANT USAGE ON AGENT PON_BIKE_DEMO.ANALYTICS.PON_BIKE_ANALYST TO ROLE PUBLIC").collect()

try:
    session.sql("""
ALTER SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT 
ADD AGENT PON_BIKE_DEMO.ANALYTICS.PON_BIKE_ANALYST
""").collect()
    print("Agent added to Snowflake Intelligence!")
except Exception as e:
    if "already present" in str(e):
        print("Agent already registered in Snowflake Intelligence, skipping.")
    else:
        raise
print()
print("Access at: https://ai.snowflake.com")

## Step 5: Test Agent

In [ ]:
def ask_agent(question: str):
    row = session.sql(f"""
    SELECT TRY_PARSE_JSON(
        SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
            'PON_BIKE_DEMO.ANALYTICS.PON_BIKE_ANALYST',
            $${{
                "messages": [
                    {{
                        "role": "user",
                        "content": [
                            {{"type": "text", "text": "{question}"}}
                        ]
                    }}
                ]
            }}$$
        )
    ) AS resp
    """).collect()[0]['RESP']
    
    if isinstance(row, str):
        result = json.loads(row)
    else:
        result = row
    
    if result and isinstance(result, dict) and 'content' in result:
        for item in result['content']:
            if isinstance(item, dict) and item.get('type') == 'text':
                return {"response": item.get('text', ''), "raw": result}
    
    return {"response": "", "raw": result}

print("Function defined - run next cell to test")

In [ ]:
result = ask_agent("How do our brand ratings compare to Trek and Specialized?")
print("Response:", result.get('response', '')[:500])
print("\nFull result:")
print(json.dumps(result.get('raw'), indent=2, default=str))

In [ ]:
test_questions = [
    # Janus (CEO)
    "How do our brand ratings compare to Trek and Specialized?",
    "Which of our brands is underperforming in the Dutch market?",
    "What would it take to get Gazelle to a 4.5 average rating?",
    # Marieke (CMO)
    "Which bike category has no strong Pon brand?",
    "What do competitor reviews praise that our reviews don't mention?",
    # Pieter (Product)
    "What are the top complaints about our e-bikes?",
    "Which product features should we prioritize based on competitor reviews?",
    "How do Cannondale road bike reviews compare to Trek and Specialized?",
    # Agent Testing
    "How is Gazelle performing compared to Batavus?",
    "What should we improve on Urban Arrow cargo bikes?",
    "Where should we invest - e-bikes or cargo bikes?"]